In [1]:
# Imports
import polars as pl
from pathlib import Path
from IPython.display import display

## Load Data

In [2]:
# Read data from parquet file, otherwise read excel file and convert to parquet
# (With polars, parquet files are much faster to read from and write to)

path_excel = Path('./5 YR 2019-2023 STATEWIDE CRASH DATA.xlsx')
path_parquet_crashes = Path('./sheet_crashes.parquet')
path_parquet_persons = Path('./sheet_persons.parquet')
sheet_name_crashes = '2019-2023 STATEWIDE'
sheet_name_persons = 'PERSONS INVOLVED'

if path_parquet_crashes.exists() and path_parquet_persons.exists():
    print(f"Reading from parquet files: {path_parquet_crashes}, {path_parquet_persons}")
    sheet_crashes_original = pl.read_parquet(path_parquet_crashes)
    sheet_persons_original = pl.read_parquet(path_parquet_persons)
    print("Done reading parquet files")
else:
    print(f"No parquet file, reading from excel file: {path_excel}")
    sheet_crashes_original = pl.read_excel(path_excel, sheet_name=sheet_name_crashes, read_options={'schema_sample_rows': None})
    sheet_persons_original = pl.read_excel(path_excel, sheet_name=sheet_name_persons, read_options={'schema_sample_rows': None})
    print(f"Done reading excel file, writing to parquet files: {path_parquet_crashes}, {path_parquet_persons}")
    sheet_crashes_original.write_parquet(path_parquet_crashes)
    sheet_persons_original.write_parquet(path_parquet_persons)
    print(f"Written to parquet files")

Reading from parquet files: sheet_crashes.parquet, sheet_persons.parquet
Done reading parquet files


## Pre-process data

### Remove Completely Identical Crash Rows

In [3]:
# Find all rows that are completely identical
sheet_crashes_original.is_duplicated().value_counts()

,count
bool,u32
false,251784
true,10


In [4]:
# Remove identical rows
sheet_crashes_no_identical = sheet_crashes_original.unique()
sheet_crashes_no_identical.is_duplicated().value_counts()

,count
bool,u32
false,251789


In [5]:
print(f"Removed: {sheet_crashes_original.shape[0] - sheet_crashes_no_identical.shape[0]}")

Removed: 5


### Remove rows with duplicate crash numbers

In [6]:
# Find all crashes with duplicate crash nums
sheet_crashes_dup = sheet_crashes_no_identical.filter(pl.col('CrashNum').is_duplicated()).sort(by='CrashNum')
sheet_crashes_dup.head()

CrashNum,Street1,Street1Other,Street1Located,Street2,Street2Other,Street2Located,MilePost,RoadPaddleMarkers,Intersection,DistLandmarkApprox,DistFromLandmark,DistFromLandmarkLocated,LandmarkUnit,DirFromLandmark,DirFromLandmarkLocated,CrashDate,CrashTime,NotifiedDate,NotifiedTime,ArrivedDate,ArrivedTime,RoadwayClearanceTime,IncidentClearanceTime,RoadIntersectType,RoadIntersectTypeOther,OfficeReport,ReportDate,ReportComplete,AgencyName,Location,OutsideTown,City,County,CountyLocated,Beat,PropertyDamage,PropertyDamageDesc,SchoolZone,PrivateProperty,TrafficControlOther,Weather,WeatherOther,RoadSurface,RoadSurfaceOther,LightCondition,LightConditionOther,CrashType,CrashSeverity,VehCrashType,AreaOfInitialImpact,LocationFirstEvent,LocationFirstEventOther,FirstHarmfulEventDesc,HitandRun,AlcoholInvolved,PedestrianInvolved,MotorcycleInvolved,NumVehicles,NumNonMotorists,NumOccupants,NumFatalities,NumInjured,NumRestrained,LocationFirstEventTurn,LocationFirstEventTrav,RoadEnvironmentalFactors,RoadEnvironFactorsOtherEnv,RoadEnvironFactorsOtherHwy,RouteID,Measure,SecondaryCollision,X,Y
str,str,str,str,str,str,str,f64,str,str,str,f64,f64,str,str,str,datetime[ms],i64,datetime[ms],i64,datetime[ms],i64,i64,i64,str,str,str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,f64,str,f64,f64
"""LVM230300114951""","""E FLAMINGO RD""",null,"""SR592""","""OTHER""","""SANDHILL RD""","""SANDHILL RD""",null,"""NONE""","""NO""","""YES""",44.0,44.0,"""FEET""","""West""","""W""",2023-03-26 00:00:00,2222,2023-03-26 00:00:00,2232,2023-03-26 00:00:00,2247,2320,2359,null,null,"""NO""",2023-03-26 00:00:00,"""YES""","""Las Vegas Metro PD""","""COUNTY""","""NO""","""LAS VEGAS""","""CLARK""","""CLARK""","""H2""","""NO""",null,"""NO""","""NO""",null,"""CLEAR|""",null,"""ASPHALT|""",null,"""DARK - CONT ROADWAY LIGHTING""",null,"""INJURY""","""C""","""REAR END""","""44 FT W/W, 28 FT N/S""","""TRAVEL LANE""",null,null,"""NO""","""YES""","""NO""","""NO""",2,0,7,0,1,7,null,2,"""NONE""",null,null,"""127CL""",8.45881,"""NO""",671713.2065,3.9984e6
"""LVM230300114951""","""E FLAMINGO RD""",null,"""SR592""","""OTHER""","""SANDHILL RD""","""SANDHILL RD""",null,"""NONE""","""NO""","""YES""",44.0,44.0,"""FEET""","""West""","""W""",2023-03-26 00:00:00,2222,2023-03-26 00:00:00,2232,2023-03-26 00:00:00,2247,2320,2359,null,null,"""NO""",2023-03-26 00:00:00,"""YES""","""Las Vegas Metro PD""","""COUNTY""","""NO""","""LAS VEGAS""","""CLARK""","""CLARK""","""H2""","""NO""",null,"""NO""","""NO""",null,"""CLEAR|""",null,"""ASPHALT|""",null,"""DARK - CONT ROADWAY LIGHTING""",null,"""INJURY""","""C""","""REAR END""","""44 FT W/W, 28 FT N/S""","""TRAVEL LANE""",null,null,"""NO""","""YES""","""NO""","""NO""",2,0,7,0,1,7,null,2,"""NONE""",null,null,"""127CL""",8.45881,null,671713.2065,3.9984e6
"""NHP221200386""","""US95""",null,"""US95S""",null,null,"""Mile Post 43""",43.0,null,"""NO""","""NO""",427.0,427.0,"""Feet""","""South""","""S""",2022-12-05 00:00:00,905,2022-12-05 00:00:00,1001,2022-12-05 00:00:00,1015,1330,1500,null,null,"""NO""",2023-07-05 00:00:00,null,"""Nevada Highway Patrol""","""County""","""YES""","""BOULDER CITY""","""CLARK""","""CLARK""","""HLJ25""","""NO""",null,"""NO""","""NO""",null,"""CLOUDY|""",null,"""ASPHALT|""",null,"""Daylight""",null,"""FATAL""","""K""","""Non-Collision""",null,"""Outside Shoulder""",null,null,"""YES""","""NO""","""NO""","""NO""",1,1,1,1,0,0,null,null,"""NONE""",null,null,"""414CL""",40.719518,"""NO""",685483.8559,3.9615e6
"""NHP221200386""","""US95""",null,"""US95S""",null,null,"""Mile Post 43""",43.0,null,"""NO""","""NO""",427.0,427.0,"""Feet""","""South""","""S""",2022-12-05 00:00:00,845,2022-12-05 00:00:00,1001,2022-12-05 00:00:00,1015,1330,1500,null,null,"""NO""",2022-12-14 00:00:00,null,"""Nevada Highway Patrol""","""County""","""YES""","""BOULDER CITY""","""CLARK""","""CLARK""","""HLJ25""","""NO""",null,"""NO""","""NO"

In [7]:
# Count ID counts
with pl.Config(tbl_rows=100):
    display(sheet_crashes_dup[:, 'CrashNum'].value_counts())

CrashNum,count
str,u32
"""LVM230300114951""",2
"""NHP221200386""",2
"""NHP221200407""",2
"""NHP230301584""",2
"""NYSO23NY-4072""",2
"""NYSO23NY-4263""",2
"""NYSO23NY-4264""",2
"""RPD23-21128""",2
"""RPD23-22397""",2


In [8]:
# Find differences between duplicate rows

def compare_pairs(df: pl.DataFrame, column_header: str, column_header2: str | None = None) -> None:
    """Print differences between each pairwise row."""
    for i in range(0, df.shape[0], 2):
        if column_header2 is not None:
            print(f"{df[i, column_header]!r}, {df[i, column_header2]!r}:")
        else:
            print(f"{df[i, column_header]!r}:")
        for column in df.columns:
            x = df[i, column]
            y = df[i+1, column]
            if x != y:
                print(f"[{i}, {column!r}]: {x!r}")
                print(f"[{i+1}, {column!r}]: {y!r}")

compare_pairs(sheet_crashes_dup, 'CrashNum')

'LVM230300114951':
[0, 'SecondaryCollision']: 'NO'
[1, 'SecondaryCollision']: None
'NHP221200386':
[2, 'CrashTime']: 905
[3, 'CrashTime']: 845
[2, 'ReportDate']: datetime.datetime(2023, 7, 5, 0, 0)
[3, 'ReportDate']: datetime.datetime(2022, 12, 14, 0, 0)
[2, 'NumOccupants']: 1
[3, 'NumOccupants']: 0
[2, 'SecondaryCollision']: 'NO'
[3, 'SecondaryCollision']: None
'NHP221200407':
[4, 'VehCrashType']: 'Unknown'
[5, 'VehCrashType']: 'SIDESWIPE - OVERTAKING'
[4, 'LocationFirstEvent']: 'Unknown'
[5, 'LocationFirstEvent']: 'TRAVEL LANE'
[4, 'LocationFirstEventTrav']: None
[5, 'LocationFirstEventTrav']: 2
[4, 'SecondaryCollision']: 'NO'
[5, 'SecondaryCollision']: None
'NHP230301584':
[6, 'DistLandmarkApprox']: 'NO'
[7, 'DistLandmarkApprox']: 'YES'
[6, 'ReportDate']: datetime.datetime(2023, 6, 20, 0, 0)
[7, 'ReportDate']: datetime.datetime(2023, 3, 21, 0, 0)
[6, 'SecondaryCollision']: 'NO'
[7, 'SecondaryCollision']: None
'NYSO23NY-4072':
[8, 'CrashDate']: datetime.datetime(2023, 11, 28, 0, 0)
[

In [9]:
# Assume that all rows with a non-null 'SecondaryCollision' value are more accurate.
# So, we will select all rows with null values and remove them
sheet_crashes_dup.filter(pl.col('SecondaryCollision').is_null())

CrashNum,Street1,Street1Other,Street1Located,Street2,Street2Other,Street2Located,MilePost,RoadPaddleMarkers,Intersection,DistLandmarkApprox,DistFromLandmark,DistFromLandmarkLocated,LandmarkUnit,DirFromLandmark,DirFromLandmarkLocated,CrashDate,CrashTime,NotifiedDate,NotifiedTime,ArrivedDate,ArrivedTime,RoadwayClearanceTime,IncidentClearanceTime,RoadIntersectType,RoadIntersectTypeOther,OfficeReport,ReportDate,ReportComplete,AgencyName,Location,OutsideTown,City,County,CountyLocated,Beat,PropertyDamage,PropertyDamageDesc,SchoolZone,PrivateProperty,TrafficControlOther,Weather,WeatherOther,RoadSurface,RoadSurfaceOther,LightCondition,LightConditionOther,CrashType,CrashSeverity,VehCrashType,AreaOfInitialImpact,LocationFirstEvent,LocationFirstEventOther,FirstHarmfulEventDesc,HitandRun,AlcoholInvolved,PedestrianInvolved,MotorcycleInvolved,NumVehicles,NumNonMotorists,NumOccupants,NumFatalities,NumInjured,NumRestrained,LocationFirstEventTurn,LocationFirstEventTrav,RoadEnvironmentalFactors,RoadEnvironFactorsOtherEnv,RoadEnvironFactorsOtherHwy,RouteID,Measure,SecondaryCollision,X,Y
str,str,str,str,str,str,str,f64,str,str,str,f64,f64,str,str,str,datetime[ms],i64,datetime[ms],i64,datetime[ms],i64,i64,i64,str,str,str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,f64,str,f64,f64
"""LVM230300114951""","""E FLAMINGO RD""",null,"""SR592""","""OTHER""","""SANDHILL RD""","""SANDHILL RD""",null,"""NONE""","""NO""","""YES""",44.0,44.0,"""FEET""","""West""","""W""",2023-03-26 00:00:00,2222,2023-03-26 00:00:00,2232,2023-03-26 00:00:00,2247,2320,2359,null,null,"""NO""",2023-03-26 00:00:00,"""YES""","""Las Vegas Metro PD""","""COUNTY""","""NO""","""LAS VEGAS""","""CLARK""","""CLARK""","""H2""","""NO""",null,"""NO""","""NO""",null,"""CLEAR|""",null,"""ASPHALT|""",null,"""DARK - CONT ROADWAY LIGHTING""",null,"""INJURY""","""C""","""REAR END""","""44 FT W/W, 28 FT N/S""","""TRAVEL LANE""",null,null,"""NO""","""YES""","""NO""","""NO""",2,0,7,0,1,7,null,2,"""NONE""",null,null,"""127CL""",8.45881,null,671713.2065,3.9984e6
"""NHP221200386""","""US95""",null,"""US95S""",null,null,"""Mile Post 43""",43.0,null,"""NO""","""NO""",427.0,427.0,"""Feet""","""South""","""S""",2022-12-05 00:00:00,845,2022-12-05 00:00:00,1001,2022-12-05 00:00:00,1015,1330,1500,null,null,"""NO""",2022-12-14 00:00:00,null,"""Nevada Highway Patrol""","""County""","""YES""","""BOULDER CITY""","""CLARK""","""CLARK""","""HLJ25""","""NO""",null,"""NO""","""NO""",null,"""CLOUDY|""",null,"""ASPHALT|""",null,"""Daylight""",null,"""FATAL""","""K""","""Non-Collision""",null,"""Outside Shoulder""",null,null,"""YES""","""NO""","""NO""","""NO""",1,1,0,1,0,0,null,null,"""NONE""",null,null,"""414CL""",40.719518,null,685483.8559,3.9615e6
"""NHP221200407""","""SR160N""",null,"""SR160N""","""LOCKSPUR AVE""",null,"""LOCKSPUR AVE""",11.0,"""NONE""","""NO""","""NO""",75.0,75.0,"""FEET""","""North""","""North""",2022-12-05 00:00:00,1501,2022-12-05 00:00:00,1503,2022-12-05 00:00:00,1508,1501,1625,null,null,"""NO""",2022-12-05 00:00:00,"""YES""","""Nevada Highway Patrol""","""COUNTY""","""YES""","""PAHRUMP""","""NYE""","""NYE""","""HLM56""","""NO""",null,"""NO""","""NO""",null,"""CLEAR|""",null,"""ASPHALT|""",null,"""DAYLIGHT""",null,"""PROPERTY""","""N""","""SIDESWIPE - OVERTAKING""",null,"""TRAVEL LANE""",null,null,"""NO""","""NO""","""NO""","""NO""",2,0,2,0,0,2,null,2,"""NONE""",null,null,"""29NY""",10.927381,null,590286.8753,4.0092e6
"""NHP230301584""","""US95S""",null,"""US95S""",null,"""SR157""","""Mile Post 93""",93.0,"""NONE""","""NO""","""YES""",892.0,892.0,"""FEET""","""South""","""S""",2023-03-21 00:00:00,1025,2023-03-21 00:00:00,1026,2023-03-21 00:00:00,1034,1330,1330,null,null,"""NO""",2023-03-21 00:00:00,"""NO""","""Nevada Highway Patrol""","""COUNTY""","""YES""",null,"""CLARK""","""CLARK""","""HLK35""","""YES""",null,"""NO""","""NO""",null,"""CLOUDY|RAIN|""",null,"""A

In [10]:
# Remove those rows
crash_nums_dup = set(sheet_crashes_dup[:, 'CrashNum'])
sheet_crashes_no_dup = sheet_crashes_no_identical.filter(pl.col('CrashNum').is_in(crash_nums_dup).not_() | pl.col('SecondaryCollision').is_not_null())
sheet_crashes_no_dup

CrashNum,Street1,Street1Other,Street1Located,Street2,Street2Other,Street2Located,MilePost,RoadPaddleMarkers,Intersection,DistLandmarkApprox,DistFromLandmark,DistFromLandmarkLocated,LandmarkUnit,DirFromLandmark,DirFromLandmarkLocated,CrashDate,CrashTime,NotifiedDate,NotifiedTime,ArrivedDate,ArrivedTime,RoadwayClearanceTime,IncidentClearanceTime,RoadIntersectType,RoadIntersectTypeOther,OfficeReport,ReportDate,ReportComplete,AgencyName,Location,OutsideTown,City,County,CountyLocated,Beat,PropertyDamage,PropertyDamageDesc,SchoolZone,PrivateProperty,TrafficControlOther,Weather,WeatherOther,RoadSurface,RoadSurfaceOther,LightCondition,LightConditionOther,CrashType,CrashSeverity,VehCrashType,AreaOfInitialImpact,LocationFirstEvent,LocationFirstEventOther,FirstHarmfulEventDesc,HitandRun,AlcoholInvolved,PedestrianInvolved,MotorcycleInvolved,NumVehicles,NumNonMotorists,NumOccupants,NumFatalities,NumInjured,NumRestrained,LocationFirstEventTurn,LocationFirstEventTrav,RoadEnvironmentalFactors,RoadEnvironFactorsOtherEnv,RoadEnvironFactorsOtherHwy,RouteID,Measure,SecondaryCollision,X,Y
str,str,str,str,str,str,str,f64,str,str,str,f64,f64,str,str,str,datetime[ms],i64,datetime[ms],i64,datetime[ms],i64,i64,i64,str,str,str,datetime[ms],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,f64,str,f64,f64
"""NHP210601588""","""IR15 N""",null,"""IR15 N""","""OTHER""","""MM19""","""OTHER""",19.0,"""BOTH SIDES""","""NO""","""YES""",500.0,500.0,"""FEET""","""North""","""North""",2021-06-21 00:00:00,818,2021-06-21 00:00:00,818,2021-06-21 00:00:00,837,908,null,null,null,"""NO""",2021-06-21 00:00:00,"""NO""","""Nevada Highway Patrol""","""COUNTY""","""YES""",null,"""CLARK""","""CLARK""","""HLI15""","""NO""",null,"""NO""","""NO""",null,"""CLEAR|""",null,"""CONCRETE|""",null,"""DAYLIGHT""",null,"""INJURY""","""A""","""REAR END""",null,"""TRAVEL LANE""",null,null,"""YES""","""NO""","""NO""","""NO""",2,0,2,0,2,1,null,2,"""NONE""",null,null,"""1CL""",19.095377,null,657113.6213,3.9694e6
"""LVM230900037014""","""S EASTERN AVE""",null,"""S EASTERN AVE""","""SUNSET RD""",null,"""SUNSET RD""",null,"""NONE""","""YES""","""NO""",null,null,"""FEET""",null,null,2023-09-11 00:00:00,954,2023-09-11 00:00:00,957,2023-09-11 00:00:00,1012,1012,1050,"""FOUR WAY""",null,"""NO""",2023-09-11 00:00:00,"""YES""","""Las Vegas Metro PD""","""COUNTY""","""NO""","""LAS VEGAS""","""CLARK""","""CLARK""","""L5""","""NO""",null,"""NO""","""NO""",null,"""CLEAR|""",null,"""ASPHALT|""",null,"""DAYLIGHT""",null,"""INJURY""","""C""","""REAR END""","""6' S/N; 10' W/E""","""INTERSECTION""",null,null,"""NO""","""NO""","""NO""","""NO""",2,0,2,0,1,2,null,null,"""NONE""",null,null,"""200299CL""",7.48879,"""NO""",669415.1768,3.9935e6
"""LVM200300038767""","""W FLAMINGO RD""",null,"""W FLAMINGO RD""","""S CIMARRON RD""",null,"""S CIMARRON RD""",null,"""NONE""","""YES""","""NO""",null,null,"""FEET""",null,null,2020-03-09 00:00:00,1213,2020-03-09 00:00:00,1248,2020-03-09 00:00:00,1316,1316,1344,"""FOUR WAY""",null,"""NO""",2020-03-09 00:00:00,"""NO""","""Las Vegas Metro PD""","""COUNTY""","""NO""","""LAS VEGAS""","""CLARK""","""CLARK""","""P1""","""NO""",null,"""NO""","""NO""",null,"""CLOUDY|""",null,"""ASPHALT|""",null,"""DAYLIGHT""",null,"""INJURY""","""C""","""ANGLE""","""22 S/N 3 E/W""","""INTERSECTION""",null,null,"""NO""","""NO""","""NO""","""NO""",2,0,2,0,1,2,null,null,"""NONE""",null,null,"""113550CL""",4.5083386,null,655700.9848,3.9981e6
"""NLPD2306060011432""","""N MARTIN L KING BLVD""",null,"""N MARTIN L KING BLVD""","""W CRAIG RD""",null,"""W CRAIG RD""",null,"""NONE""","""NO""","""YES""",750.0,750.0,"""FEET""","""South""","""S""",2023-06-06 00:00:00,735,2023-06-06 00:00:00,738,2023-06-06 00:00:00,741,null,null,null,null,"""NO""",2023-06-06 00:00:00,"""YES""","""North Las Vegas PD""","""CITY""","""NO""","""NORTH LAS VEGAS""","""CLARK""","""CLARK""","""G2""","""NO""",null,"""NO""","

In [11]:
sheet_crashes_no_dup[:, 'CrashNum'].is_duplicated().value_counts()

CrashNum,count
bool,u32
false,251773


In [12]:
print("===Counts===")
print(f"                                Identical: {sheet_crashes_original.is_duplicated().value_counts().row(by_predicate=pl.col('') == True)[1]}")
print(f"        Duplicate CrashNum, not identical: {sheet_crashes_dup.shape[0]}")
print(f"                        Identical removed: {sheet_crashes_original.shape[0] - sheet_crashes_no_identical.shape[0]}")
print(f"Duplicate CrashNum, not identical Removed: {sheet_crashes_no_identical.shape[0] - sheet_crashes_no_dup.shape[0]}")
print(f"                            Total Removed: {sheet_crashes_original.shape[0] - sheet_crashes_no_dup.shape[0]}")

===Counts===
                                Identical: 10
        Duplicate CrashNum, not identical: 32
                        Identical removed: 5
Duplicate CrashNum, not identical Removed: 16
                            Total Removed: 21


### Remove Completely Identical Persons Rows

In [13]:
sheet_persons_original[:, ['CrashNum', 'pplNo']].is_duplicated().value_counts()

,count
bool,u32
true,52
false,705429


In [14]:
sheet_persons_original.is_duplicated().value_counts()

,count
bool,u32
true,42
false,705439


In [15]:
sheet_persons_no_identical = sheet_persons_original.unique()
print(f"{sheet_persons_original.shape = }")
print(f"{sheet_persons_no_identical.shape = }")
print(f"Removed: {sheet_persons_original.shape[0] - sheet_persons_no_identical.shape[0]}")
sheet_persons_no_identical.is_duplicated().value_counts()

sheet_persons_original.shape = (705481, 84)
sheet_persons_no_identical.shape = (705460, 84)
Removed: 21


,count
bool,u32
false,705460


### Remove persons rows with duplicate CrashNum, pplNo pairs

In [16]:
# Find all crashes with duplicate crash nums
sheet_persons_dup = sheet_persons_no_identical.filter(sheet_persons_no_identical[:, ['CrashNum', 'pplNo']].is_duplicated()).sort(by=('CrashNum', 'pplNo'))
sheet_persons_dup

CrashNum,pplNo,vcNoRelated,PersonType,VehUnit,Age,Sex,Race,Driver,VehOwner,DLClass,DLCommercial,DLStatus,NumberOfCitations,PersonFactors,PersonFactInatt,AlcDrugInvolved,SuspectedImpairment,DrugSpecimen1,DrugSpecimen2,DrugResults,DrugResults2,DrugResults3,FieldSobrietyTest,UrineBloodRes,AirBag,AirbagSwitch,RestraintUsed,Helmet,Ejected,Trapped,SeatPosition,InjuryCode,InjuryLocation1,InjuryLocation2,InjuryLocation3,DeathDate,…,NMNumVehStrike3,NonMotorist,NMTypeOther,NMCondition,NMConditionOther,NMSpeedLow,NMSpeedHigh,NMSpeedLimit,NMActions,NMActionsOther,NMBikeLane,NMBikeLaneOther,NMPedDamage,VehicleDamage1stCon,VehicleDamageRating1,NMSafetyEquip,NMSafetyEquipOther,NMFactor,NMFactorOther,VehicleType,VehicleMake,VehicleModel,VehicleSerialNo,VehicleSeq1Code,VehicleSeq2Code,VehicleSeq3Code,VehicleSeq4Code,VehicleSeq5Code,VehicleSeq1CFO,VehicleSeq2CFO,VehicleSeq3CFO,VehicleSeq4CFO,VehicleSeq5CFO,VehicleSeq1Harm,NMVehActions,NMVehActionsOther,NonMotorDamageArea
str,i64,i64,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,date,…,i64,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""RPD23-22397""",82071025,79435511,"""Driver""","""1""",57,"""M""",null,"""YES""","""NO""","""C""","""NO""","""Suspended""",0,"""APPARENTLY NORMAL;""",null,"""NONE""","""NO""","""FIELD SOBRIETY TEST""","""BLOOD TEST""",null,null,null,"""PASS""","""NONE""","""Not Deployed""","""Unknown Switch Position""","""Used Shoulder and Lap Belt""",null,"""Not Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""O""",null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""RPD23-22397""",82071025,79435511,"""Driver""","""1""",57,"""M""",null,"""YES""","""NO""","""C""","""NO""","""Suspended""",0,"""APPARENTLY NORMAL;""",null,"""UNKNOWN""","""NO""",null,null,null,null,null,null,null,"""Not Deployed""","""Unknown Switch Position""","""Used Shoulder and Lap Belt""",null,"""Not Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""O""",null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""RPD23-22397""",82071026,79435512,"""Driver""","""2""",70,"""M""",null,"""YES""","""NO""","""A""","""YES""","""Valid""",0,"""UNKNOWN;""",null,"""UNKNOWN""","""NO""",null,null,null,null,null,null,null,"""Not Applicable""","""ON-OFF Switch not Present""","""Not Applicable""","""Other Helmet""","""Totally Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""K""",1,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""RPD23-22397""",82071026,79435512,"""Driver""","""2""",70,"""M""",null,"""YES""","""NO""","""A""","""YES""","""Valid""",0,"""DRUG INVOLVEMENT;""",null,"""DRUGS""","""NO""","""BLOOD TEST""",null,null,null,null,null,"""1000 NG""","""Not Applicable""","""ON-OFF Switch not Present""","""Not Applicable""","""Other Helmet""","""Totally Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""K""",1,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""RPD23-23007""",82285813,0,"""Pedestrian""","""1""",40,"""F""",null,"""NO""","""NO""",null,null,null,0,null,null,"""UNKNOWN""","""NO""",null,null,null,null,null,null,null,null,null,null,null,null,"""Not Trapped""",null,"""K""",4,null,null,null,…,null,"""YES""",null,"""UNKNOWN""",null,null,null,null,"

In [17]:
compare_pairs(sheet_persons_dup, 'CrashNum', 'pplNo')

'RPD23-22397', 82071025:
[0, 'AlcDrugInvolved']: 'NONE'
[1, 'AlcDrugInvolved']: 'UNKNOWN'
[0, 'DrugSpecimen1']: 'FIELD SOBRIETY TEST'
[1, 'DrugSpecimen1']: None
[0, 'DrugSpecimen2']: 'BLOOD TEST'
[1, 'DrugSpecimen2']: None
[0, 'FieldSobrietyTest']: 'PASS'
[1, 'FieldSobrietyTest']: None
[0, 'UrineBloodRes']: 'NONE'
[1, 'UrineBloodRes']: None
'RPD23-22397', 82071026:
[2, 'PersonFactors']: 'UNKNOWN;'
[3, 'PersonFactors']: 'DRUG INVOLVEMENT;'
[2, 'AlcDrugInvolved']: 'UNKNOWN'
[3, 'AlcDrugInvolved']: 'DRUGS'
[2, 'DrugSpecimen1']: None
[3, 'DrugSpecimen1']: 'BLOOD TEST'
[2, 'UrineBloodRes']: None
[3, 'UrineBloodRes']: '1000 NG'
'RPD23-23007', 82285813:
[4, 'AlcDrugInvolved']: 'UNKNOWN'
[5, 'AlcDrugInvolved']: 'DRUGS'
[4, 'DrugSpecimen1']: None
[5, 'DrugSpecimen1']: 'BLOOD TEST'
[4, 'UrineBloodRes']: None
[5, 'UrineBloodRes']: 'FAIL'
[4, 'NMLocPrior2Imp']: 'UNKNOWN'
[5, 'NMLocPrior2Imp']: 'In Roadway'
[4, 'NMCondition']: 'UNKNOWN'
[5, 'NMCondition']: 'Under Influence: Med/Drugs/Alcohol'
[4, '

In [18]:
# Manually remove data (Removing data if it has more nulls)
sheet_persons_no_dup = sheet_persons_no_identical.clone()
sheet_persons_no_dup = sheet_persons_no_dup.remove(pl.col('CrashNum') == 'RPD23-22397', pl.col('pplNo') == 82071025, pl.col('DrugSpecimen1').is_null())
sheet_persons_no_dup = sheet_persons_no_dup.remove(pl.col('CrashNum') == 'RPD23-22397', pl.col('pplNo') == 82071026, pl.col('DrugSpecimen1').is_null())
sheet_persons_no_dup = sheet_persons_no_dup.remove(pl.col('CrashNum') == 'RPD23-23007', pl.col('pplNo') == 82285813, pl.col('DrugSpecimen1').is_null())
sheet_persons_no_dup = sheet_persons_no_dup.remove(pl.col('CrashNum') == 'SPPD23-9197', pl.col('pplNo') == 81902340, pl.col('PersonFactors') == 'Apparently Normal')  # Equal number of nulls
sheet_persons_no_dup = sheet_persons_no_dup.remove(pl.col('CrashNum') == 'SPPD23-9197', pl.col('pplNo') == 81902341, pl.col('Sex').is_null())
sheet_persons_no_dup

CrashNum,pplNo,vcNoRelated,PersonType,VehUnit,Age,Sex,Race,Driver,VehOwner,DLClass,DLCommercial,DLStatus,NumberOfCitations,PersonFactors,PersonFactInatt,AlcDrugInvolved,SuspectedImpairment,DrugSpecimen1,DrugSpecimen2,DrugResults,DrugResults2,DrugResults3,FieldSobrietyTest,UrineBloodRes,AirBag,AirbagSwitch,RestraintUsed,Helmet,Ejected,Trapped,SeatPosition,InjuryCode,InjuryLocation1,InjuryLocation2,InjuryLocation3,DeathDate,…,NMNumVehStrike3,NonMotorist,NMTypeOther,NMCondition,NMConditionOther,NMSpeedLow,NMSpeedHigh,NMSpeedLimit,NMActions,NMActionsOther,NMBikeLane,NMBikeLaneOther,NMPedDamage,VehicleDamage1stCon,VehicleDamageRating1,NMSafetyEquip,NMSafetyEquipOther,NMFactor,NMFactorOther,VehicleType,VehicleMake,VehicleModel,VehicleSerialNo,VehicleSeq1Code,VehicleSeq2Code,VehicleSeq3Code,VehicleSeq4Code,VehicleSeq5Code,VehicleSeq1CFO,VehicleSeq2CFO,VehicleSeq3CFO,VehicleSeq4CFO,VehicleSeq5CFO,VehicleSeq1Harm,NMVehActions,NMVehActionsOther,NonMotorDamageArea
str,i64,i64,str,str,i64,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,date,…,i64,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""NLPD2305020008770""",76799597,74560714,"""Driver""","""1""",18,"""M""",null,"""YES""","""NO""","""C""","""NO""","""Normal with Restrictions""",0,"""DRIVER INATTENTION/DISTRACTED""","""Unknown""","""NONE""","""NO""",null,null,null,null,null,null,null,"""Not Deployed""","""ON-OFF Switch not Present""","""Used Shoulder and Lap Belt""",null,"""Not Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""O""",null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""NLPD2309090018101""",80316555,0,"""Witness""","""A""",null,"""M""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""LVM230900006936""",79856825,0,"""Witness""","""A""",22,"""F""",null,"""NO""","""NO""",null,null,null,0,null,null,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""NHP231100682""",81520887,78929115,"""Driver""","""1""",33,"""F""",null,"""YES""","""NO""","""C""","""NO""","""Valid""",0,"""APPARENTLY NORMAL;""",null,null,"""NO""",null,null,null,null,null,null,null,"""Deployed Front""","""Unknown if ON-OFF Switch Prese…","""Used Shoulder and Lap Belt""","""No Helmet""","""Not Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""O""",null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""CCSO2021-4773""",63464021,62107080,"""Driver""","""1""",null,"""M""",null,null,null,"""C""","""NO""","""Valid""",-1,"""UNKNOWN""",null,"""UNKNOWN""","""NO""",null,null,null,null,null,null,null,"""Not Deployed""","""Unknown Switch Position""","""Not Applicable""","""No Helmet""","""Not Ejected""","""Not Trapped""","""1st Row - Left Side (Motorcycl…","""N""",null,null,null,null,…,null,"""NO""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,

In [19]:
print(f"{sheet_persons_no_identical.shape = }")
print(f"{sheet_persons_no_dup.shape = }")
print(f"Removed: {sheet_persons_no_identical.shape[0] - sheet_persons_no_dup.shape[0]}")

sheet_persons_no_identical.shape = (705460, 84)
sheet_persons_no_dup.shape = (705455, 84)
Removed: 5


### Convert to Uppercase

In [20]:
sheet_crashes_no_dup = sheet_crashes_no_dup.with_columns(pl.col('County').str.to_uppercase())
with pl.Config(tbl_rows=100):
    display(sheet_crashes_no_dup[:, 'County'].value_counts())

County,count
str,u32
"""CHURCHILL""",2049
"""WASHOE""",35841
"""CARSON CITY""",4342
"""DOUGLAS""",2899
"""CLARK""",188258
"""EUREKA""",499
"""LANDER""",509
"""NV""",6
"""PERSHING""",549


## Analysis

### Join tables

In [21]:
print(f"{sheet_crashes_no_dup.shape = }")

sheet_crashes_no_dup.shape = (251773, 74)


In [22]:
# Check null counts
print(f"{sheet_crashes_no_dup[:, 'CrashNum'].null_count() = }")
print(f"{sheet_persons_no_dup[:, 'CrashNum'].null_count() = }")
print(f"{sheet_persons_no_dup[:, 'pplNo'].null_count()    = }")

sheet_crashes_no_dup[:, 'CrashNum'].null_count() = 0
sheet_persons_no_dup[:, 'CrashNum'].null_count() = 0
sheet_persons_no_dup[:, 'pplNo'].null_count()    = 0


In [23]:
# Check that the set of crash numbers from the persons table is a subset of
# of the set of crash numbers from the crashes table
crash_nums_crashes = set(sheet_crashes_no_dup[:, 'CrashNum'])
crash_nums_persons = set(sheet_persons_no_dup[:, 'CrashNum'])

print(f"{len(crash_nums_crashes) = }")
print(f"{len(crash_nums_persons) = }")
print(f"{crash_nums_crashes.issubset(crash_nums_persons) = }")
print(f"{crash_nums_persons.issubset(crash_nums_crashes) = }")

len(crash_nums_crashes) = 251773
len(crash_nums_persons) = 251506
crash_nums_crashes.issubset(crash_nums_persons) = False
crash_nums_persons.issubset(crash_nums_crashes) = True


In [24]:
# Join tables
sheet_joined = sheet_crashes_no_dup.join(sheet_persons_no_dup, on='CrashNum', how='left')
print(f"Joined table shape: {sheet_joined.shape}")
print(f"Unpaired rows: {sheet_joined[:, 'pplNo'].null_count()}")

Joined table shape: (705722, 157)
Unpaired rows: 267


### Does every row with PedestrianInvolved always include persons marked with "WheelChair"?

In [25]:
sheet_crashes_no_dup[:, 'PedestrianInvolved'].value_counts()

PedestrianInvolved,count
str,u32
"""YES""",4810
null,136
"""NO""",246827


In [26]:
with pl.Config(tbl_rows=100):
    display(sheet_persons_no_dup[:, 'PersonType'].value_counts().sort('count', descending=True))

PersonType,count
str,u32
"""Driver""",478985
"""Passenger""",173408
"""Witness""",44202
"""Pedestrian""",5127
"""Pedal Cyclist""",2331
"""Unknown""",223
"""Other Non-Motorist""",202
"""Other""",187
"""Wheelchair""",163


In [27]:
# Find rows with pedestrian
sheet_joined.filter(pl.col('PedestrianInvolved') == 'NO', pl.col('PersonType') == 'Wheelchair')

CrashNum,Street1,Street1Other,Street1Located,Street2,Street2Other,Street2Located,MilePost,RoadPaddleMarkers,Intersection,DistLandmarkApprox,DistFromLandmark,DistFromLandmarkLocated,LandmarkUnit,DirFromLandmark,DirFromLandmarkLocated,CrashDate,CrashTime,NotifiedDate,NotifiedTime,ArrivedDate,ArrivedTime,RoadwayClearanceTime,IncidentClearanceTime,RoadIntersectType,RoadIntersectTypeOther,OfficeReport,ReportDate,ReportComplete,AgencyName,Location,OutsideTown,City,County,CountyLocated,Beat,PropertyDamage,…,NMNumVehStrike3,NonMotorist,NMTypeOther,NMCondition,NMConditionOther,NMSpeedLow,NMSpeedHigh,NMSpeedLimit,NMActions,NMActionsOther,NMBikeLane,NMBikeLaneOther,NMPedDamage,VehicleDamage1stCon,VehicleDamageRating1,NMSafetyEquip,NMSafetyEquipOther,NMFactor,NMFactorOther,VehicleType,VehicleMake,VehicleModel,VehicleSerialNo,VehicleSeq1Code,VehicleSeq2Code,VehicleSeq3Code,VehicleSeq4Code,VehicleSeq5Code,VehicleSeq1CFO,VehicleSeq2CFO,VehicleSeq3CFO,VehicleSeq4CFO,VehicleSeq5CFO,VehicleSeq1Harm,NMVehActions,NMVehActionsOther,NonMotorDamageArea
str,str,str,str,str,str,str,f64,str,str,str,f64,f64,str,str,str,datetime[ms],i64,datetime[ms],i64,datetime[ms],i64,i64,i64,str,str,str,datetime[ms],str,str,str,str,str,str,str,str,str,…,i64,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""LVM190700007066""","""S BRUCE ST""",null,"""S BRUCE ST""","""E CHARLESTON BLVD""",null,"""E CHARLESTON BLVD""",null,"""NONE""","""NO""","""YES""",null,null,"""FEET""","""North""","""North""",2019-07-02 00:00:00,1002,2019-07-02 00:00:00,1012,2019-07-02 00:00:00,1021,1002,1121,null,null,"""NO""",2019-07-02 00:00:00,"""NO""","""Las Vegas Metro PD""","""CITY""","""NO""","""LAS VEGAS""","""CLARK""","""CLARK""","""A5""","""NO""",…,null,"""YES""",null,"""APPARENTLY NORMAL""",null,null,null,null,"""ENTERING OR CROSSING AT LOCATI…",null,null,null,"""LEFT SIDE""",null,null,"""None""",null,"""3|""",null,null,null,null,null,"""Motor Vehicle In Transport""",null,null,null,null,"""YES""","""NO""","""NO""","""NO""","""NO""","""1""",null,null,null
"""BCPD19BC0757""","""US93 S""",null,"""US93 S""","""NEVADA WAY""",null,"""NEVADA WAY""",null,null,"""NO""","""YES""",26.0,26.0,"""Feet""","""East""","""East""",2019-06-20 00:00:00,855,2019-06-20 00:00:00,857,2019-06-20 00:00:00,903,928,935,null,null,"""NO""",2019-06-20 00:00:00,null,"""Boulder City PD""","""City""","""NO""","""BOULDER CITY""",null,null,"""BC1""","""NO""",…,null,"""NO""",null,"""Apparently Normal""",null,2,5,"""25""","""Entering or Crossing at Locati…",null,"""No Bike Lane Path""",null,"""Front""","""Front""","""Unknown""","""None""",null,null,null,"""NONE""","""SILVER PRIDE""","""RAPTOR""",null,"""Motor Vehicle In Transport""",null,null,null,null,"""NO""","""NO""","""NO""","""NO""","""NO""","""1""","""Straight""",null,"""UNKNOWN|"""
"""NLPD1902090002905""","""N LAS VEGAS BLVD""",null,"""N LAS VEGAS BLVD""","""CIVIC CENTER DR""",null,"""CIVIC CENTER DR""",null,"""NONE""","""NO""","""YES""",18.0,18.0,"""FEET""","""North""","""North""",2019-02-09 00:00:00,1806,2019-02-09 00:00:00,1807,2019-02-09 00:00:00,1807,null,null,null,null,"""NO""",2019-02-09 00:00:00,"""NO""","""North Las Vegas PD""","""CITY""","""NO""","""NORTH LAS VEGAS""","""CLARK""","""CLARK""",null,"""NO""",…,null,"""YES""",null,"""UNKNOWN""",null,null,null,"""35""","""Entering or Crossing at Locati…",null,null,null,"""RIGHT SIDE""",null,"""Total""","""None""",null,null,null,null,null,null,null,"""Motor Vehicle In Transport""",null,null,null,null,"""YES""","""NO""","""NO""","""NO""","""NO""","""1""","""Straight""",null,"""FRONT|"""
"""LVM211100019485""","""S DECATUR BLVD""",null,"""S DECATUR BLVD""","""DOE AVE""",null,"""DOE AVE""",null,"""NONE""","""NO""","""YES""",2.0,2.0,"""FEET""","""North""","""North""",2021-11-05 00:00:00,1450,2021-11-05 00:00:00,1512,2021-11-05 00:00:00,1534,1534,1605,null,null,"""NO""",2021-11-05 00:00:00,"""NO""","""Las Vegas Metro PD""","""

Answer: Non-empty table, so yes

## Extract Pedestrian Data

In [28]:
# Get counts to know how many rows we are expecting.
group_by = sheet_joined.group_by('CrashNum')
any_pedestrian = (pl.col('PersonType') == 'Pedestrian').any()
any_wheelchair = (pl.col('PersonType') == 'Wheelchair').any()
any_pedestrian_involved_null = pl.col('PedestrianInvolved').is_null().any()
any_pedestrian_involved_yes = (pl.col('PedestrianInvolved') == 'YES').any()
any_pedestrian_involved_no = (pl.col('PedestrianInvolved') == 'NO').any()

print("PedestrianInvolved=null:")
print(group_by.agg(any_pedestrian_involved_null & any_pedestrian.not_() & any_wheelchair.not_())[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_null & any_pedestrian        & any_wheelchair.not_())[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_null & any_pedestrian.not_() & any_wheelchair       )[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_null & any_pedestrian        & any_wheelchair       )[:, 1].sum())

print("PedestrianInvolved=YES:")
print(group_by.agg(any_pedestrian_involved_yes & any_pedestrian.not_() & any_wheelchair.not_())[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_yes & any_pedestrian        & any_wheelchair.not_())[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_yes & any_pedestrian.not_() & any_wheelchair       )[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_yes & any_pedestrian        & any_wheelchair       )[:, 1].sum())

print("PedestrianInvolved=NO:")
print(group_by.agg(any_pedestrian_involved_no & any_pedestrian.not_() & any_wheelchair.not_())[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_no & any_pedestrian        & any_wheelchair.not_())[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_no & any_pedestrian.not_() & any_wheelchair       )[:, 1].sum())
print(group_by.agg(any_pedestrian_involved_no & any_pedestrian        & any_wheelchair       )[:, 1].sum())

PedestrianInvolved=null:
136
0
0
0
PedestrianInvolved=YES:
4
4802
0
4
PedestrianInvolved=NO:
246665
4
158
0


In [29]:
# Get crash numbers
crash_nums_pedestrian = set(group_by.agg(any_pedestrian_involved_yes | any_pedestrian | any_wheelchair).filter(pl.col('PedestrianInvolved'))[:, 'CrashNum'])
crash_nums_pedestrian

{'LVM230300003533',
 'LVM211000049473',
 'NHP220501640',
 'LVM230900093024',
 'NHP210200806',
 'RPD19-599',
 'RPD20-17533',
 'RPD20-19432',
 'LVM201000054121',
 'LVM211100082240',
 'LVM230400069166',
 'LVM200500038930',
 'LVM210400103842',
 'LVM221100083376',
 'RPD20-15924',
 'LVM231200034553',
 'NLPD200706010941',
 'LVM210200024954',
 'LVM220800065947',
 'LVM220100028323',
 'ELPD2020-18238',
 'LVM201100102878',
 'LVM230100000591',
 'CLSD22052657',
 'LVM191100028103',
 'HP23-12126',
 'LVM210800081467',
 'LVM230200103630',
 'LVM230800083213',
 'NLPD2011140018903',
 'LVM211000083258',
 'LVM210900007883',
 'LVM210200090076',
 'LVM221100024868',
 'LVM190600137330',
 'SPPD22-9568',
 'NHP231200079',
 'RPD22-16797',
 'LVM210700030792',
 'LVM191100020280',
 'LVM190900132218',
 'WASO23-5138',
 'NLPD2308300017401',
 'LVM200200107091',
 'LVM221200016065',
 'WASD19W-00746',
 'LVM231200039816',
 'LVM191100142184',
 'NLPD2107280012788',
 'LVM220900016421',
 'LVM211200107609',
 'WASO22-5037',
 'LVM19

In [30]:
sorted(sheet_crashes_no_dup.columns)

['AgencyName',
 'AlcoholInvolved',
 'AreaOfInitialImpact',
 'ArrivedDate',
 'ArrivedTime',
 'Beat',
 'City',
 'County',
 'CountyLocated',
 'CrashDate',
 'CrashNum',
 'CrashSeverity',
 'CrashTime',
 'CrashType',
 'DirFromLandmark',
 'DirFromLandmarkLocated',
 'DistFromLandmark',
 'DistFromLandmarkLocated',
 'DistLandmarkApprox',
 'FirstHarmfulEventDesc',
 'HitandRun',
 'IncidentClearanceTime',
 'Intersection',
 'LandmarkUnit',
 'LightCondition',
 'LightConditionOther',
 'Location',
 'LocationFirstEvent',
 'LocationFirstEventOther',
 'LocationFirstEventTrav',
 'LocationFirstEventTurn',
 'Measure',
 'MilePost',
 'MotorcycleInvolved',
 'NotifiedDate',
 'NotifiedTime',
 'NumFatalities',
 'NumInjured',
 'NumNonMotorists',
 'NumOccupants',
 'NumRestrained',
 'NumVehicles',
 'OfficeReport',
 'OutsideTown',
 'PedestrianInvolved',
 'PrivateProperty',
 'PropertyDamage',
 'PropertyDamageDesc',
 'ReportComplete',
 'ReportDate',
 'RoadEnvironFactorsOtherEnv',
 'RoadEnvironFactorsOtherHwy',
 'RoadEnv

In [31]:
# Filter and extract data
columns_extract_crashes = [
    'CrashNum',
    'Street1',
    'Street1Other',
    'Street1Located',
    'Street2',
    'Street2Other',
    'Street2Located',
    'CrashDate',
    'CrashTime',
    'City',
    'County',
    'CountyLocated',
    'X',
    'Y',
]
sheet_crashes_pedestrian = sheet_crashes_no_dup.filter(pl.col('CrashNum').is_in(crash_nums_pedestrian))[:, columns_extract_crashes]
sheet_crashes_pedestrian

CrashNum,Street1,Street1Other,Street1Located,Street2,Street2Other,Street2Located,CrashDate,CrashTime,City,County,CountyLocated,X,Y
str,str,str,str,str,str,str,datetime[ms],i64,str,str,str,f64,f64
"""RPD23-6918""","""NORTH HILLS BLVD""",null,"""NORTH HILLS BLVD""","""E GOLDEN VALLEY RD""",null,"""E GOLDEN VALLEY RD""",2023-04-19 00:00:00,1415,"""RENO""","""WASHOE""","""WASHOE""",256127.4965,4.3877e6
"""LVM190700007066""","""S BRUCE ST""",null,"""S BRUCE ST""","""E CHARLESTON BLVD""",null,"""E CHARLESTON BLVD""",2019-07-02 00:00:00,1002,"""LAS VEGAS""","""CLARK""","""CLARK""",668657.2255,4.0032e6
"""CCSO21-7927""","""N STEWART ST""",null,"""N STEWART ST""","""E ROBINSON ST""",null,"""E ROBINSON ST""",2021-12-03 00:00:00,1835,"""CARSON CITY""","""CARSON CITY""","""CARSON CITY""",261167.4269,4.3390e6
"""LVM211100036560""","""W CRAIG RD""",null,"""W CRAIG RD""","""OTHER""","""RAINBOW""","""OTHER""",2021-11-09 00:00:00,1701,"""LAS VEGAS""","""CLARK""","""CLARK""",658062.3295,4.0120e6
"""NHP220101574""","""SR372""",null,"""SR372""","""PAHRUMP VALLEY BLVD""",null,"""PAHRUMP VALLEY BLVD""",2022-01-24 00:00:00,1546,"""PAHRUMP""","""NYE""","""NYE""",590697.804,4.0073e6
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HP21-02090""","""ST ROSE PKWY""",null,"""ST ROSE PKWY""","""ALPER CENTER DR""",null,"""ALPER CENTER DR""",2021-02-04 00:00:00,215,"""HENDERSON""","""CLARK""","""CLARK""",668408.0021,3.9850e6
"""LVM220300046344""","""E HARMON AVE""",null,"""E HARMON AVE""","""AUDRIE ST""","""AUDRIE""","""AUDRIE ST""",2022-03-11 00:00:00,2048,"""LAS VEGAS""","""CLARK""","""CLARK""",664925.3528,3.9975e6
"""NLPD1907200014915""","""E CAREY AVE""",null,"""E CAREY AVE""","""N LAS VEGAS BLVD""",null,"""N LAS VEGAS BLVD""",2019-07-20 00:00:00,1050,"""NORTH LAS VEGAS""","""CLARK""","""CLARK""",669188.5296,4.0081e6


## Extract Bicyclist Data

In [32]:
# Get crash numbers
any_pedal_cyclist = (pl.col('PersonType') == 'Pedal Cyclist').any()
crash_nums_bicyclist = set(group_by.agg(any_pedal_cyclist).filter(pl.col('PersonType'))[:, 'CrashNum'])
crash_nums_bicyclist

{'LVM200500045594',
 'LVM230900087323',
 'WASO21-3153',
 'RPD23-4665',
 'LVM231000037631',
 'FAPDTA19-0063',
 'LVM220300049695',
 'LVM200100016962',
 'LVM191000112151',
 'LVM221000103802',
 'SPPD23-4209',
 'NHP210501221',
 'LVM230800087029',
 'RPD20-7291',
 'RPD21-8137',
 'LVM220700065076',
 'HP19-19036',
 'RPD21-9796',
 'HP21-20884',
 'NLPD2109020015018',
 'LVM220200086266',
 'LVM201000117476',
 'LVM190400036283',
 'LVM210600119896',
 'NHP200600419',
 'LVM191000051600',
 'NHP211101500',
 'RPD20-14070',
 'RPD23-17265',
 'SPPD23-7662',
 'LVM211100083173',
 'RPD22-14183',
 'LVM210200024550',
 'NLPD2302070002620',
 'LVM220900034982',
 'RPD20-11764',
 'LVM230500048561',
 'NLPD2309200018806',
 'LVM191200022327',
 'RPD19-17656',
 'LVM190500003879',
 'LVM220500099215',
 'HP20-13121',
 'LVM210700003340',
 'ELPD2019-21546',
 'LVM200700093565',
 'CHSOC20-05990',
 'HP22-15090',
 'LVM211100001832',
 'WPSO2107140030',
 'CCSO23-5255',
 'RPD19-19955',
 'RPD20-16776',
 'CCSO21-5716',
 'LVM190500146418

In [33]:
sheet_crashes_bicyclist = sheet_crashes_no_dup.filter(pl.col('CrashNum').is_in(crash_nums_bicyclist))[:, columns_extract_crashes]
sheet_crashes_bicyclist

CrashNum,Street1,Street1Other,Street1Located,Street2,Street2Other,Street2Located,CrashDate,CrashTime,City,County,CountyLocated,X,Y
str,str,str,str,str,str,str,datetime[ms],i64,str,str,str,f64,f64
"""LVM210300001603""","""ARVILLE ST""",null,"""ARVILLE ST""","""PENNWOOD AVE""",null,"""PENNWOOD AVE""",2021-03-01 00:00:00,948,"""LAS VEGAS""","""CLARK""","""CLARK""",662005.1902,4.0007e6
"""NLPD2306140012098""","""BENNETT ST""",null,"""BENNETT ST""","""GILDAY AVE""",null,"""GILDAY AVE""",2023-06-14 00:00:00,1608,"""NORTH LAS VEGAS""","""CLARK""","""CLARK""",666778.2052,4.0075e6
"""NLPD2101030000128""","""W CHEYENNE AVE""",null,"""W CHEYENNE AVE""","""N DECATUR BLVD""",null,"""N DECATUR BLVD""",2021-01-03 00:00:00,941,"""NORTH LAS VEGAS""","""CLARK""","""CLARK""",661491.988,4.0096e6
"""SPPD19-6925""","""E PRATER WY""",null,"""E PRATER WY""","""PULLMAN DR""",null,"""PULLMAN DR""",2019-08-04 00:00:00,930,"""SPARKS""",null,null,266410.5307,4.3804e6
"""NHP210200759""","""OTHER""","""IR80/ E. MCCARRRAN BLVD OFF""","""OTHER""","""OTHER""","""MCCARRAN BLVD""","""OTHER""",2021-02-12 00:00:00,1237,"""SPARKS""","""WASHOE""","""WASHOE""",253948.9066,4.3788e6
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HP23-05355""","""N GREEN VALLEY PKWY""",null,"""WARM SPRINGS RD""","""W WARM SPRINGS RD""",null,"""N GREEN VALLEY PKWY""",2023-03-22 00:00:00,1349,"""HENDERSON""","""CLARK""","""CLARK""",672559.8804,3.9919e6
"""RPD21-13841""","""W MOANA LN""",null,"""W MOANA LN""","""KIETZKE LN""",null,"""KIETZKE LN""",2021-07-31 00:00:00,2127,"""RENO""","""WASHOE""","""WASHOE""",260074.871,4.3752e6
"""LVM200100012625""","""S RAINBOW BLVD""",null,"""S RAINBOW BLVD""","""W TROPICANA AVE""",null,"""W TROPICANA AVE""",2020-01-03 00:00:00,1259,"""LAS VEGAS""","""CLARK""","""CLARK""",658160.0777,3.9963e6


## Write Data to file

In [34]:
path_out_pedestrian = Path('./crashes_pedestrian.csv')
path_out_bicyclist  = Path('./crashes_bicyclist.csv')

sheet_crashes_pedestrian.write_csv(path_out_pedestrian)
sheet_crashes_bicyclist.write_csv(path_out_bicyclist)

print("Wrote data.")

Wrote data.
